# Observatorio de Obras Públicas · 02 Limpieza

Toma `datos/obras.csv` (lo que trajo el paso 01) y lo deja listo para analizar:

1. Arregla textos, fechas y enlaces.
2. Quita los contratos que SECOP marca como obra pero no lo son.
3. Corrige el municipio donde **realmente** se hace la obra.
4. Une los nombres repetidos de un mismo contratista.
5. Clasifica cada obra por sector (agua, vías, educación...) y por modalidad de contratación.
6. Calcula cuánto subió el valor de cada contrato con las adiciones.
7. Marca **alertas** para revisar a mano.
8. Limpia los procesos adjudicados sin contrato publicado (`datos/sin_contrato.csv`).

El resultado se guarda en `datos/obras_limpias.csv`, `datos/excluidos.csv` y `datos/sin_contrato_limpio.csv`.

**Cómo ejecutarlo:** pon el cursor en una celda y pulsa **Shift + Enter**. Se ejecuta esa celda y pasa a la siguiente.

## 1 · Herramientas y datos

Carga las librerías y lee el archivo del paso 01. Al final muestra cuántos contratos y columnas hay.

In [1]:
import re, ast, unicodedata
import pandas as pd
import duckdb

# Números grandes sin decimales; porcentajes pequeños con un decimal
pd.options.display.float_format = lambda x: f"{x:,.0f}" if abs(x) >= 100 else f"{x:,.1f}"
pd.options.display.max_colwidth = 80

obras = pd.read_csv("datos/obras.csv")
print(f"{len(obras):,} contratos y {obras.shape[1]} columnas")
obras.head(3)

368 contratos y 17 columnas


,id,departamento,municipio,tipo,fecha_firma,objeto,valor,valor_con_adiciones,entidad,contratista,fecha_inicio,fecha_fin,dias_adicionados,estado,modalidad,url,fuente
0,CO1.PCCNTR.9466943,Cauca,GUAPI,Obra,2026-04-30 11:34:59.694,REALIZAR LAS OBRAS DE ADECUACIONES MENORES EN LAS UNIDADES DE ATENCIÓN EN SA...,"2,361,210,192",NaN,EMPRESA SOCIAL DEL ESTADO GUAPI ESE,CONSORCIO PUESTOS DE SALUD GUAPI,2026-05-18T00:00:00.000,2026-10-24T23:59:00.000,0,En ejecución,Contratación régimen especial (con ofertas),{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/I...,SECOP II
1,CO1.PCCNTR.8736766,Cauca,TIMBIQUI,Obra,2026-01-20 20:41:43.047,REPOSICIÓN DE LA INFRAESTRUCTURA FISÍCA DE LA UNIDAD NIVEL I LOPEZ DE MICAY;...,"42,374,296,130",NaN,EMPRESA SOCIAL DE ESTADO OCCIDENTE,CONSORCIO SEIKO,2026-02-12T08:00:00.000,2028-08-12T23:59:00.000,0,En ejecución,Contratación régimen especial (con ofertas),{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/I...,SECOP II
2,CO1.PCCNTR.8504659,Cauca,TIMBIQUI,Obra,2025-10-30 11:18:23.189,ADECUACIONES MENORES EN LA INFRAESTRUCTURA DE LA UNIDAD DE DOS QUEBRADAS; ES...,"216,625,105",NaN,EMPRESA SOCIAL DE ESTADO OCCIDENTE,CONSTRUCTORA DEL PACIFICO CONPACIF SAS,2025-12-01T00:00:00.000,2025-12-31T23:59:00.000,0,En ejecución,Contratación régimen especial (con ofertas),{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/I...,SECOP II


## 2 · ¿Qué datos faltan?

Cuenta las celdas vacías de cada columna. Es normal que `dias_adicionados` esté casi vacía: SECOP I no la publica.

In [2]:
vacios = obras.isna().sum()
vacios[vacios > 0].to_frame("vacíos")

,vacíos
valor_con_adiciones,6


## 3 · Textos, fechas y enlaces

- **Textos:** SECOP I guarda las tildes como minúsculas sueltas (`CONSTRUCCIoN`). Se pasa todo a mayúsculas sin tildes (`CONSTRUCCION`) para poder buscar palabras.
- **Fechas:** se convierten a fecha real y se calcula el **plazo** en días.
- **Enlaces:** en SECOP II el enlace viene dentro de `{'url': ...}`; se deja solo la dirección.

In [3]:
def normalizar(texto):
    """Mayúsculas, sin tildes y sin espacios sobrantes: 'Construcción ' -> 'CONSTRUCCION'."""
    if pd.isna(texto):
        return ""
    texto = unicodedata.normalize("NFKD", str(texto).upper())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(texto.split())


def limpiar_url(valor):
    """"{'url': 'https://...'}" -> 'https://...'"""
    if isinstance(valor, str) and valor.startswith("{"):
        try:
            return ast.literal_eval(valor).get("url", "")
        except (ValueError, SyntaxError):
            return ""
    return valor


obras["objeto"] = obras["objeto"].map(normalizar)
obras["url"] = obras["url"].map(limpiar_url)
for c in ("fecha_firma", "fecha_inicio", "fecha_fin"):
    obras[c] = pd.to_datetime(obras[c], errors="coerce").dt.normalize()
obras["anio"] = obras["fecha_firma"].dt.year
obras["plazo_dias"] = (obras["fecha_fin"] - obras["fecha_inicio"]).dt.days

obras[["objeto", "fecha_firma", "plazo_dias", "url"]].head(3)

,objeto,fecha_firma,plazo_dias,url
0,REALIZAR LAS OBRAS DE ADECUACIONES MENORES EN LAS UNIDADES DE ATENCION EN SA...,2026-04-30,159,https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noti...
1,REPOSICION DE LA INFRAESTRUCTURA FISICA DE LA UNIDAD NIVEL I LOPEZ DE MICAY;...,2026-01-20,912,https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noti...
2,ADECUACIONES MENORES EN LA INFRAESTRUCTURA DE LA UNIDAD DE DOS QUEBRADAS; ES...,2025-10-30,30,https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noti...


## 3.1 · Contratos que no son obra

Algunas entidades marcan en SECOP como *Obra* contratos que no lo son, por ejemplo una prestación de servicios de apoyo a la gestión para las juntas de acción comunal. Se excluyen por la descripción:

- **Prestación de servicios:** empieza por *PRESTACION DE SERVICIOS PROFESIONALES* o *DE APOYO*, o habla de *APOYO A LA GESTION* o *APOYO LOGISTICO*. Un "servicio de mantenimiento a todo costo" de un edificio sí se deja, porque es obra.
- **Interventoría:** el contrato es la interventoría (la supervisión) de una obra, no la obra.

Los excluidos no se borran: se guardan en `datos/excluidos.csv` con el motivo, para poder revisarlos.

In [4]:
NO_OBRA = {
    "Prestación de servicios": r"^PRESTACION DE SERVICIOS (?:PROFESIONALES|DE APOYO)|APOYO A LA GESTION|APOYO LOGISTICO",
    "Interventoría":           r"^(?:EL CONTRATISTA SE OBLIGA .*? A )?(?:REALIZAR )?(?:LA )?INTERVENTORIA",
}
obras["motivo_exclusion"] = ""
for motivo, patron in NO_OBRA.items():
    coincide = obras["objeto"].str.contains(patron, regex=True) & obras["motivo_exclusion"].eq("")
    obras.loc[coincide, "motivo_exclusion"] = motivo

excluidos = obras[obras["motivo_exclusion"] != ""].copy()
obras = obras[obras["motivo_exclusion"] == ""].drop(columns="motivo_exclusion").reset_index(drop=True)
print(f"{len(excluidos)} contratos excluidos por no ser obra · quedan {len(obras):,}")
excluidos[["municipio", "valor", "motivo_exclusion", "objeto"]]

7 contratos excluidos por no ser obra · quedan 361


,municipio,valor,motivo_exclusion,objeto
5,TIMBIQUI,"1,775,577,641",Interventoría,EL CONTRATISTA SE OBLIGA CON LA EMPRESA SOCIAL DEL ESTADO ESE OCCIDENTE A RE...
16,LOPEZ DE MICAY,"10,506,000",Prestación de servicios,PRESTACION DE SERVICIOS DE APOYO A LA GESTION COMO AUXILIAR A LA INSPECCION ...
28,LOPEZ DE MICAY,"29,999,000",Prestación de servicios,PRESTACION DE SERVICIOS DE APOYO LOGISTICO PARA LA REALIZACION DE MESAS DE G...
42,LOPEZ DE MICAY,"17,290,000",Prestación de servicios,PRESTACION DE SERVICIOS PROFESIONALES DE APOYO A LA SECRETARIA DE GOBIERNO E...
95,LOPEZ DE MICAY,"11,028,000",Prestación de servicios,PRESTACION DE SERVICIOS DE APOYO A LA GESTION EN TAREAS RELACIONADAS CON EL ...
188,LOPEZ DE MICAY,"10,500,000",Prestación de servicios,PRESTACION DE SERVICIOS DE APOYO A LA GESTION EN TAREAS RELACIONADAS CON LA ...
322,GUAPI,"8,000,000",Prestación de servicios,PRESTACION DE SERVICIOS DE APOYO A LA GESTION COMO ENLACE PARA LA ARTICULACI...


## 4 · Municipio donde se hace la obra

SECOP guarda el municipio de la **entidad que contrata**, no el de la obra. Por ejemplo, la E.S.E. Occidente tiene sede en Timbiquí pero contrató el hospital de López de Micay.

Regla: si la descripción del contrato menciona **un solo** municipio de la lista, ese es el municipio de la obra. Si menciona varios o ninguno, se deja el de la entidad.

In [5]:
MUNICIPIOS = ["GUAPI", "TIMBIQUI", "LOPEZ DE MICAY"]

def municipio_obra(fila):
    mencionados = [m for m in MUNICIPIOS if m in fila["objeto"]]
    return mencionados[0] if len(mencionados) == 1 else fila["municipio"]

obras["municipio_entidad"] = obras["municipio"]
obras["municipio"] = obras.apply(municipio_obra, axis=1)

cambiados = obras[obras["municipio"] != obras["municipio_entidad"]]
print(f"{len(cambiados)} contratos cambiaron de municipio:")
cambiados[["municipio_entidad", "municipio", "valor", "objeto"]]

1 contratos cambiaron de municipio:


,municipio_entidad,municipio,valor,objeto
1,TIMBIQUI,LOPEZ DE MICAY,"42,374,296,130",REPOSICION DE LA INFRAESTRUCTURA FISICA DE LA UNIDAD NIVEL I LOPEZ DE MICAY;...


## 5 · Unir los nombres de un mismo contratista

El mismo contratista aparece escrito de varias formas: `VC&P INGENIERIA S.A.S.` y `VC&P INGENIERIA S.A.S`.

1. Se quitan puntos, comas y la sigla `SAS`.
2. Si un nombre limpio es el comienzo de otro (`HARNOLD FAVIAN RIASCOS` y `HARNOLD FAVIAN RIASCOS RIASCOS`), se toman como la misma persona y se usa el nombre más frecuente.

Revisa la tabla de abajo: si alguna unión está mal, avísame y se ajusta la regla.

In [6]:
def nombre_base(nombre):
    n = normalizar(nombre)
    n = re.sub(r"[.,]", " ", n)
    n = re.sub(r"\bS\s*A\s*S\b", " ", n)
    return " ".join(n.split())

obras["contratista_base"] = obras["contratista"].map(nombre_base)

# Del más frecuente al menos frecuente: cada nombre se une al primero que lo contenga al inicio
canonico = {}
for nombre in obras["contratista_base"].value_counts().index:
    canonico[nombre] = next((c for c in set(canonico.values())
                             if nombre.startswith(c) or c.startswith(nombre)), nombre)
obras["contratista_unico"] = obras["contratista_base"].map(canonico)

uniones = (obras.groupby("contratista_unico")["contratista"].unique()
                .loc[lambda s: s.str.len() > 1])
print(f"{obras['contratista'].nunique()} nombres originales -> {obras['contratista_unico'].nunique()} contratistas")
uniones.to_frame("nombres que se unieron")

102 nombres originales -> 89 contratistas


,nombres que se unieron
contratista_unico,
GARCIA INGENIERO,"[GARCIA INGENIEROS SAS, GARCIA INGENIERO S.A.S]"
HARNOLD FAVIAN RIASCOS RIASCOS,"[HARNOLD FAVIAN RIASCOS RIASCOS, HARNOLD FAVIAN RIASCOS]"
IVAN ENRIQUE TORRES ZUNIGA,"[IVAN ENRIQUE TORRES ZUÑIGA, IVAN ENRIQUE TORRES]"
JOSE NEIVER RODRIGUEZ,"[JOSE NEIVER RODRIGUEZ ARBOLEDA, JOSE NEIVER RODRIGUEZ]"
P&B INGENIERIA,"[P&B INGENIERIA S.A.S, P&B INGENIERIA SAS]"
R&R INGENIERIA Y ARQUITECTURA,"[R&R INGENIERIA Y ARQUITECTURA S.A.S., R&R INGENIERIA Y ARQUITECTURA SAS]"
SOLUCIONES ECOAMBIENTALES Y EMPRESARIALES,"[SOLUCIONES ECOAMBIENTALES Y EMPRESARIALES S.A.S SECOEM, SOLUCIONES ECOAMBIE..."
VC&P INGENIERIA,"[VC&P INGENIERIA S.A.S., VC&P INGENIERIA S.A.S, VC&P INGENIERIAS, VC&P INGEN..."
YADER EVACIO PAREDES VIDAL,"[YADER EVACIO PAREDES VIDAL, YADER EVACIO PAREDES]"


## 6 · Sector de cada obra

Se busca palabras clave en la descripción. Se revisa en orden y gana el primer sector que coincida (por eso salud va antes que edificios: un hospital es salud, no un edificio cualquiera; y deporte va antes que vías, porque la dirección de un parque suele nombrar una *CALLE*).

Cada palabra clave tiene que empezar una palabra de la descripción. Así *BIOSALUDABLE* no se confunde con *SALUD*, ni *BARRIO* o *TERRITORIO* con *RIO*. Lo que queda en **Otro** se revisó a mano y se agregaron palabras para pistas de aterrizaje, esteros, caminos y edificios municipales.

In [7]:
SECTORES = {
    "Salud":                  ["HOSPITAL", "SALUD", "E.S.E", "PUESTO DE SALUD", "UNIDAD DE ATENCION"],
    "Educación":              ["AULA", "ESCUELA", "COLEGIO", "EDUCATIV", "RESTAURANTE ESCOLAR"],
    "Agua y saneamiento":     ["ACUEDUCTO", "ALCANTARILLADO", "AGUA POTABLE", "SANEAMIENTO", "POZO", "TANQUE"],
    "Quebradas y ríos":       ["QUEBRADA", "RIO ", "CAUCE", "DRAGADO", "MURO DE CONTENCION", "ESTERO", "CANAL", "CANOS"],
    "Deporte y recreación":   ["ESTADIO", "CANCHA", "POLIDEPORTIVO", "PARQUE", "DEPORTIV"],
    "Vías y transporte":      ["VIA", "VIAL", "PUENTE", "MUELLE", "SENDERO", "ANDEN", "PAVIMENT", "CALLE", "CARRETERA",
                               "CAMINO", "EMBARCADERO", "SALTADERO", "AEROPUERTO", "AERODROMO", "PISTA DE ATERRIZAJE",
                               "REDUCTORES DE VELOCIDAD", "SENALIZACION"],
    "Energía y alumbrado":    ["ELECTRIC", "ALUMBRADO", "ENERGIA", "SOLAR"],
    "Vivienda":               ["VIVIENDA"],
    "Edificios públicos":     ["SEDE ADMINISTRATIVA", "PLAZA DE MERCADO", "ALCALDIA", "CASA", "CEMENTERIO", "SALON",
                               "CONCEJO", "PERSONERIA", "CENTRO ADMINISTRATIVO", "BIBLIOTECA", "CARCEL", "CENTRO CULTURAL",
                               "INFRAESTRUCTURA CULTURAL", "EDIFICIO", "SEDE", "BATALLON", "HOGAR", "ADULTO MAYOR",
                               "INMUEBLE", "SACUDETE"],
    "Espacio público":        ["PLAZOLETA", "ESPACIO PUBLICO"],
    "Productivo y agro":      ["AVICOLA", "PRODUCTIV", "PROCUTIV", "TRAPICHE", "AGROPECUAR", "PISCICOLA"],
}
# Cada palabra clave debe empezar una palabra: así "BIOSALUDABLE" no cuenta como "SALUD"
PATRONES = {nombre: re.compile(r"\b(?:" + "|".join(map(re.escape, palabras)) + ")")
            for nombre, palabras in SECTORES.items()}

def sector(objeto):
    for nombre, patron in PATRONES.items():
        if patron.search(objeto):
            return nombre
    return "Otro"

obras["sector"] = obras["objeto"].map(sector)
obras["sector"].value_counts().to_frame("obras")

,obras
sector,
Quebradas y ríos,147
Vías y transporte,79
Agua y saneamiento,56
Educación,23
Deporte y recreación,21
Edificios públicos,19
Salud,5
Productivo y agro,3
Espacio público,3


## 6.1 · Modalidad de contratación

SECOP I y SECOP II escriben distinto la modalidad (`Contratación Mínima Cuantía` y `Mínima cuantía`). Se agrupan en las modalidades de la ley:

| Modalidad | Qué es |
|---|---|
| **Mínima cuantía** | Contratos pequeños (menos del 10 % de la menor cuantía de la entidad). Basta una invitación pública de un día. |
| **Selección abreviada** | Contratos medianos, con un proceso más corto que la licitación. |
| **Licitación pública** | El proceso más abierto y competido; obligatorio para obras grandes. |
| **Contratación directa** | Sin competencia: la entidad escoge al contratista. Solo se permite en casos que la ley enumera. |
| **Régimen especial** | Entidades como las E.S.E. de salud, que contratan con su propio manual. |

Es uno de los indicadores de transparencia más usados: mucha contratación directa o muchos contratos de mínima cuantía pueden señalar un contrato grande partido en pedazos para evitar la licitación.

In [8]:
def modalidad(texto):
    t = normalizar(texto)
    if "MINIMA CUANTIA" in t:
        return "Mínima cuantía"
    if "DIRECTA" in t:
        return "Contratación directa"
    if "REGIMEN ESPECIAL" in t:
        return "Régimen especial"
    if "SELECCION ABREVIADA" in t:
        return "Selección abreviada"
    if "LICITACION" in t:
        return "Licitación pública"
    if "CONCURSO" in t:
        return "Concurso de méritos"
    return "Otra" if t else "Sin dato"

obras["modalidad"] = obras["modalidad"].map(modalidad)
duckdb.sql("""
    SELECT modalidad, COUNT(*) AS obras, SUM(valor) AS valor_total,
           ROUND(100 * SUM(valor) / SUM(SUM(valor)) OVER (), 1) AS porcentaje_del_valor
    FROM obras GROUP BY modalidad ORDER BY obras DESC
""").df()

,modalidad,obras,valor_total,porcentaje_del_valor
0,Mínima cuantía,256,"8,253,192,941",7.0
1,Selección abreviada,67,"15,402,770,350",13.1
2,Licitación pública,23,"34,798,956,867",29.7
3,Contratación directa,10,"5,449,302,313",4.6
4,Régimen especial,5,"53,441,355,410",45.5


## 7 · Adiciones: cuánto subió cada contrato

`valor` es lo que se firmó y `valor_con_adiciones` es lo que terminó costando. La diferencia es la **adición** (en pesos y en %).

**Ojo:** en estos municipios SECOP I publica `valor_con_adiciones` igual al valor firmado, y SECOP II no lo publica. Por eso hoy da 0. La celda se deja lista para cuando cambies de región o los datos se actualicen.

In [9]:
obras["valor_con_adiciones"] = obras["valor_con_adiciones"].fillna(obras["valor"])
obras["adicion"] = obras["valor_con_adiciones"] - obras["valor"]
obras["adicion_pct"] = (obras["adicion"] / obras["valor"] * 100).round(1)

con_adicion = obras[obras["adicion"] > 0]
print(f"{len(con_adicion)} contratos tuvieron adiciones")
con_adicion[["municipio", "contratista_unico", "valor", "valor_con_adiciones", "adicion_pct"]] \
    .sort_values("adicion_pct", ascending=False)

0 contratos tuvieron adiciones


,municipio,contratista_unico,valor,valor_con_adiciones,adicion_pct


## 8 · Alertas para revisar

No son pruebas de nada; son contratos que **merecen una mirada** a mano:

| Alerta | Qué significa |
|---|---|
| `valor_atipico` | Vale más de 20 veces lo que cuesta una obra típica (la mediana) en ese municipio. |
| `objeto_repetido` | El mismo tipo de obra contratado 5 veces o más en el mismo municipio y año. Podría ser un contrato grande partido en pedazos. |
| `sin_firmar` | En SECOP I figura como *Convocado*: el proceso se abrió pero aún no hay contrato. |
| `adicion_alta` | El valor subió más de 30 % con las adiciones. |

In [10]:
obras["valor_atipico"] = obras.groupby("municipio")["valor"].transform(lambda v: v > 20 * v.median())

# "Tipo de obra" = primeras 6 palabras de la descripción, sin las frases de relleno
# como "EL CONTRATISTA SE OBLIGA CON EL MUNICIPIO A REALIZAR..."
RELLENO = r"^(EL CONTRATISTA SE OBLIGA (PARA )?CON .*? A |CONTRATAR |REALIZAR |EJECUTAR )*(LA |LAS |LOS |EL )?"
obras["objeto_corto"] = (obras["objeto"].str.replace(RELLENO, "", regex=True)
                                        .str.split().str[:6].str.join(" "))
repeticiones = obras.groupby(["municipio", "anio", "objeto_corto"])["id"].transform("size")
obras["objeto_repetido"] = repeticiones >= 5

obras["sin_firmar"] = obras["estado"].eq("Convocado")
obras["adicion_alta"] = obras["adicion_pct"] > 30

ALERTAS = ["valor_atipico", "objeto_repetido", "sin_firmar", "adicion_alta"]
obras["n_alertas"] = obras[ALERTAS].sum(axis=1)
obras[ALERTAS].sum().to_frame("contratos")

,contratos
valor_atipico,10
objeto_repetido,148
sin_firmar,53
adicion_alta,0


### 8.1 · Obras repetidas

Cuántas veces se contrató el mismo tipo de obra en el mismo municipio y año, y cuántos contratistas distintos la hicieron.

In [11]:
duckdb.sql("""
    SELECT municipio, anio, objeto_corto,
           COUNT(*)                          AS contratos,
           COUNT(DISTINCT contratista_unico) AS contratistas,
           SUM(valor)                        AS valor_total,
           ROUND(AVG(valor))                 AS valor_promedio
    FROM obras
    WHERE objeto_repetido
    GROUP BY ALL
    ORDER BY contratos DESC
""").df()

,municipio,anio,objeto_corto,contratos,contratistas,valor_total,valor_promedio
0,LOPEZ DE MICAY,2022,OBRAS DE MEJORAMIENTO; MANTENIMIENTO Y LIMPIEZA,57,2,"1,577,170,273","27,669,654"
1,LOPEZ DE MICAY,2023,OBRAS DE MEJORAMIENTO; MANTENIMIENTO Y LIMPIEZA,42,2,"1,358,164,241","32,337,244"
2,LOPEZ DE MICAY,2024,OBRAS DE MEJORAMIENTO; MANTENIMIENTO Y LIMPIEZA,22,3,"784,793,350","35,672,425"
3,LOPEZ DE MICAY,2024,OBRAS PARA LA OPTIMIZACION DEL ACUEDUCTO,14,2,"470,645,350","33,617,525"
4,LOPEZ DE MICAY,2025,OBRAS DE MANTENIMIENTO Y LIMPIEZA DE,7,1,"889,349,009","127,049,858"
5,LOPEZ DE MICAY,2024,OBRAS DE MANTENIMIENTO Y LIMPIEZA DE,6,1,"218,312,000","36,385,333"


### 8.2 · Contratos con valor atípico

In [12]:
obras[obras["valor_atipico"]][["municipio", "anio", "sector", "contratista_unico", "valor", "objeto"]] \
    .sort_values("valor", ascending=False)

,municipio,anio,sector,contratista_unico,valor,objeto
1,LOPEZ DE MICAY,2026,Salud,CONSORCIO SEIKO,"42,374,296,130",REPOSICION DE LA INFRAESTRUCTURA FISICA DE LA UNIDAD NIVEL I LOPEZ DE MICAY;...
3,TIMBIQUI,2026,Salud,CONSORCIO OCCIDENTE CAUCA,"8,311,996,833",EL CONTRATISTA SE OBLIGA CON LA E.S.E. A REALIZAR LAS ADECUACIONES MENORES E...
255,GUAPI,2022,Deporte y recreación,CONSORCIO ESTADIO GUAPI 2022,"7,330,324,782",CONSTRUCCION Y ADECUACION DE LA PRIMERA ETAPA DEL ESTADIO DEL MUNICIPIO DE G...
314,LOPEZ DE MICAY,2022,Edificios públicos,CONSORCIO MICAY 2022,"5,406,693,215",REALIZAR LOS ESTUDIOS; DISENOS Y OBRAS DE CONSTRUCCION DE LA SEDE ADMINISTRA...
235,LOPEZ DE MICAY,2022,Educación,UNION TEMPORAL HJB,"4,178,196,325",CONTRATAR LAS OBRAS PARA LA EJECUCION DEL CONVENIO NUMERO 3063 DE 2021 CUYO ...
205,GUAPI,2026,Agua y saneamiento,IVAN ENRIQUE TORRES ZUNIGA,"2,690,523,194",REALIZAR LOS ESTUDIOS; DISENOS Y OPTIMIZACION DE LOS ACUEDUCTOS DE LOS CORRE...
0,GUAPI,2026,Salud,CONSORCIO PUESTOS DE SALUD GUAPI,"2,361,210,192",REALIZAR LAS OBRAS DE ADECUACIONES MENORES EN LAS UNIDADES DE ATENCION EN SA...
222,LOPEZ DE MICAY,2025,Agua y saneamiento,UNION TEMPORAL PTAP MICAY 2025,"1,400,000,000",OPTIMIZACION Y MEJORAMIENTO DE LA PLANTA DE TRATAMIENTO DE AGUA POTABLE - PT...
352,LOPEZ DE MICAY,2022,Vías y transporte,UNION TEMPORAL VIAS LOPEZ,"881,078,211",MANTENIMIENTO DE VIAS INTERNAS EN LA CABECERA MUNICIPAL DE LOPEZ DE MICAY - ...
112,LOPEZ DE MICAY,2025,Agua y saneamiento,HARNOLD FAVIAN RIASCOS RIASCOS,"670,795,632",CONSTRUCCION ACUEDUCTO RURAL DEL CORREGIMIENTO DE SAN ANTONIO DE GURUMENDI D...


## 9 · Resúmenes

### 9.1 · Por sector

In [13]:
duckdb.sql("""
    SELECT sector,
           COUNT(*)   AS obras,
           SUM(valor) AS valor_total
    FROM obras
    GROUP BY sector
    ORDER BY valor_total DESC
""").df()

,sector,obras,valor_total
0,Salud,5,"53,441,355,410"
1,Quebradas y ríos,147,"13,390,763,532"
2,Agua y saneamiento,56,"12,708,498,006"
3,Vías y transporte,79,"11,173,380,289"
4,Deporte y recreación,21,"9,800,900,048"
5,Educación,23,"8,927,066,049"
6,Edificios públicos,19,"7,212,314,235"
7,Otro,2,"269,770,563"
8,Espacio público,3,"199,479,882"
9,Productivo y agro,3,"108,246,567"


### 9.2 · Contratistas con más obras

`porcentaje_del_municipio` dice qué parte de todo lo contratado en su municipio se llevó ese contratista.

In [14]:
duckdb.sql("""
    WITH por_contratista AS (
        SELECT municipio, contratista_unico,
               COUNT(*) AS obras, SUM(valor) AS valor_total
        FROM obras GROUP BY ALL
    )
    SELECT *,
           ROUND(100 * valor_total / SUM(valor_total) OVER (PARTITION BY municipio), 1)
               AS porcentaje_del_municipio
    FROM por_contratista
    ORDER BY obras DESC
    LIMIT 10
""").df()

,municipio,contratista_unico,obras,valor_total,porcentaje_del_municipio
0,LOPEZ DE MICAY,VC&P INGENIERIA,83,"2,459,833,488",3.4
1,LOPEZ DE MICAY,SOLUCIONES ECOAMBIENTALES Y EMPRESARIALES,50,"4,493,812,919",6.1
2,LOPEZ DE MICAY,HARNOLD FAVIAN RIASCOS RIASCOS,45,"4,294,834,552",5.9
3,LOPEZ DE MICAY,JENER GARCIA CARABALI,29,"791,341,476",1.1
4,LOPEZ DE MICAY,YADER EVACIO PAREDES VIDAL,10,"318,703,756",0.4
5,GUAPI,VC&P INGENIERIA,9,"857,718,073",3.0
6,GUAPI,NILSON RICARDO TORRES OBREGON,7,"369,250,403",1.3
7,LOPEZ DE MICAY,CARLOS EMIRO RIASCOS MANCILLA,5,"160,067,671",0.2
8,LOPEZ DE MICAY,DIDIER JOEL VELLAIZAC TORRES,5,"382,568,800",0.5
9,LOPEZ DE MICAY,PAREDES GLOBAL,5,"230,208,000",0.3


## 10 · Procesos adjudicados sin contrato publicado

En SECOP I hay procesos de obra que figuran como **Adjudicados** pero no tienen fecha de firma, contratista ni valor del contrato: la entidad nunca subió el contrato. Su único valor es el **presupuesto del proceso**.

No se suman a la inversión, porque no se sabe si el contrato se firmó ni por cuánto. Pero se muestran aparte: si un municipio tiene muchos, su inversión firmada aparece más baja de lo que probablemente es y la comparación por habitante queda sesgada.

Se limpian con las mismas reglas que los contratos.

In [15]:
sin_contrato = pd.read_csv("datos/sin_contrato.csv")
sin_contrato["objeto"] = sin_contrato["objeto"].map(normalizar)
sin_contrato["url"] = sin_contrato["url"].map(limpiar_url)
sin_contrato["fecha_cargue"] = pd.to_datetime(sin_contrato["fecha_cargue"], errors="coerce").dt.normalize()
sin_contrato["anio"] = sin_contrato["fecha_cargue"].dt.year
sin_contrato["municipio_entidad"] = sin_contrato["municipio"]
sin_contrato["municipio"] = [municipio_obra(f) for _, f in sin_contrato.iterrows()]   # también sirve si viene vacía
sin_contrato["sector"] = sin_contrato["objeto"].map(sector)
sin_contrato["modalidad"] = sin_contrato["modalidad"].map(modalidad)

duckdb.sql("""
    SELECT s.municipio,
           COUNT(*)            AS procesos_sin_contrato,
           SUM(s.presupuesto)  AS presupuesto,
           ANY_VALUE(f.valor)  AS valor_contratos_firmados,
           ROUND(SUM(s.presupuesto) / ANY_VALUE(f.valor), 2) AS veces_lo_firmado
    FROM sin_contrato s
    JOIN (SELECT municipio, SUM(valor) AS valor FROM obras GROUP BY municipio) f USING (municipio)
    GROUP BY s.municipio ORDER BY presupuesto DESC
""").df()

,municipio,procesos_sin_contrato,presupuesto,valor_contratos_firmados,veces_lo_firmado
0,TIMBIQUI,54,"31,513,059,370","16,124,103,770",1.9
1,LOPEZ DE MICAY,47,"20,639,621,866","73,076,629,793",0.3
2,GUAPI,11,"3,510,225,190","28,144,844,318",0.1


## 11 · Guardar

Guarda tres tablas, sin las columnas de trabajo intermedias:

- `datos/obras_limpias.csv`: los contratos de obra que usa el tablero.
- `datos/excluidos.csv`: los que SECOP marca como obra pero no lo son, con el motivo.
- `datos/sin_contrato_limpio.csv`: los procesos adjudicados sin contrato publicado.

In [16]:
COLUMNAS = ["id", "fuente", "municipio", "municipio_entidad", "anio", "fecha_firma",
            "fecha_inicio", "fecha_fin", "plazo_dias", "sector", "modalidad", "objeto",
            "entidad", "contratista", "contratista_unico", "estado",
            "valor", "valor_con_adiciones", "adicion", "adicion_pct",
            *ALERTAS, "n_alertas", "url"]

obras[COLUMNAS].to_csv("datos/obras_limpias.csv", index=False)
excluidos[["id", "fuente", "municipio", "fecha_firma", "valor", "entidad", "contratista", "motivo_exclusion", "objeto", "url"]] \
    .to_csv("datos/excluidos.csv", index=False)
sin_contrato[["id", "proceso", "fuente", "municipio", "municipio_entidad", "anio", "fecha_cargue", "sector",
              "modalidad", "objeto", "entidad", "presupuesto", "url"]].to_csv("datos/sin_contrato_limpio.csv", index=False)
print(f"Guardado: datos/obras_limpias.csv ({len(obras):,} contratos, {len(COLUMNAS)} columnas)")
print(f"Guardado: datos/excluidos.csv ({len(excluidos)} contratos que no son obra)")
print(f"Guardado: datos/sin_contrato_limpio.csv ({len(sin_contrato)} procesos adjudicados sin contrato)")

Guardado: datos/obras_limpias.csv (361 contratos, 26 columnas)
Guardado: datos/excluidos.csv (7 contratos que no son obra)
Guardado: datos/sin_contrato_limpio.csv (112 procesos adjudicados sin contrato)
